# sele_pdb and all of its helper functions

In [1]:
from pathlib import Path
import sys
sys.path.append( str( Path("../../../.." ).resolve()) )

- ## sele_closest_Chain

In [ ]:
# Load test datasets
import gemmi
from xaidar.data.molecModels import flatten_pdb
from xaidar.data.molecModels import get_pdb_stats, get_res_CoM
from xaidar.data.molecModels import  sele_pdb, sele_Lig, sele_AA
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")

lig_pdb = sele_pdb( ev2a_pdb, sele_Lig )
lig_com = get_res_CoM( flatten_pdb( lig_pdb, "residue") )
lig_com = gemmi.Position( *lig_com[0] )

ev2a_pdb = sele_pdb( ev2a_pdb, sele_AA)
# get_pdb_stats( ev2a_pdb)

In [ ]:
def sele_closest_Chain( lst_chains: list[gemmi.Chain], 
                       CoM: gemmi.Position ,    level = False,
                       verbose = False) -> list[gemmi.Chain]:
    """
    Select the chain that contains the ligand and is closest to the CoM
    of the protein.
    Args:
    - lst_chains (list[gemmi.Chain]): List of chains in the PDB.
    - CoM (gemmi.Position): Center of Mass of the protein.
    - ligLabel (str, optional): Ligand residue name. Defaults to "LIG".
    Returns:
    - list[gemmi.Chain]: List containing the selected chain.
    """
    
    if level: return "chain"
    if len(lst_chains) == 1:
        if verbose: print("Only one chain in the PDB, returning it")
        return lst_chains
    else:
        # Select the chain closest to the CoM
        min_dist = float('inf')
        selected_chain = None
        for chain in lst_chains:
            dist = chain.calculate_center_of_mass().dist(CoM)
            if dist < min_dist:
                min_dist = dist
                selected_chain = chain
        return [selected_chain]

In [ ]:
# Test function
get_pdb_stats( ev2a_pdb )
prot = sele_pdb( ev2a_pdb, sele_closest_Chain, lig_com )
get_pdb_stats( prot )


####################
Number of models: 1
Number of chains in 1st Model: 2

Chain ID: A
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.
Chain ID: B
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.

####################
Number of models: 1
Number of chains in 1st Model: 1

Chain ID: B
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.


- ## sele_res_idx()

In [8]:
# load test pdbs

import gemmi
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")
cox2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/coxb4_2a.pdb")
ev2a1_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a_1.pdb")
from xaidar.data.protocols import protein_processing
from xaidar.data.molecModels import sele_pdb, sele_AA, sele_model
from xaidar.data.molecModels import get_pdb_stats
ev2a_prot = protein_processing(ev2a_pdb)
ev2a1_prot = protein_processing(ev2a1_pdb)
cox2a_prot = cox2a_pdb
for foo in [ sele_AA, sele_model]:
    cox2a_prot = sele_pdb(cox2a_prot, foo)
# get_pdb_stats(ev2a_prot)
# get_pdb_stats(ev2a1_prot)
# get_pdb_stats(cox2a_prot)

In [9]:
import gemmi
from xaidar.data.molecModels import sele_pdb

def sele_res_idx( lst_res: list[gemmi.Residue], lst_slices: list[ tuple ],
                slice_within = True, level = False  ):
    """
    Remove residues from a list based on specified slices.
    Args:
    - lst_res (list[gemmi.Residue]): List of residue objects.
    - lst_slices (list[tuple]): List of tuples specifying slices to remove.
        Must be in (start, end) format, where 'start' is inclusive and 'end' is exclusive.
        The slices should not overlap. The integers must be positive and within the range of lst_res.
    - slice_within (bool, optional): If True, remove residues within the slices.
        If False, remove residues outside the slices. Defaults to True.
    - level (bool, optional): If True, operate at residue level. Defaults to False.
    Returns:
    - list[gemmi.Residue]: Updated list of residue objects after removal.
    """
    if level: return "residue"
    lst_slices = sorted( lst_slices, reverse=True )
    new_lst_res = lst_res if slice_within else []
    for slice_idx in lst_slices:
        start, end = slice_idx
        if slice_within:
            del new_lst_res[start:end]
        else:
            new_lst_res = new_lst_res + lst_res[start:end]
    return new_lst_res

In [14]:
# Test
from xaidar.data.molecModels import get_chain_seq, model_seqAlign

from copy import deepcopy
refprot = ev2a_prot
queryprot = cox2a_prot
gap_query = deepcopy( queryprot)
gap_query_seq = get_chain_seq( gap_query )[0]
print( gap_query_seq )

gap_query = sele_pdb( gap_query, sele_res_idx, [  (10,20) ] ) # testing removing residues 10-19
gap_query_seq = get_chain_seq( gap_query )[0]
print( gap_query_seq )

alignment = model_seqAlign( refprot, queryprot,).visualize().map_matching_res( match_type = "exact", gaps = True)
print( "\t###########################################################")
alignment = model_seqAlign( refprot, gap_query,).visualize().map_matching_res( match_type = "exact", gaps = True)



GPYGHQSGAVYVGNYKVVNRHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
GPYGHQSGAVHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
                |         |         |         |         *         |         |         |         |         +         |         |         |         |         *
Ref:   ------SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLD----EE
             |||:|||||:|||||||||.||.|.||||.:||||||:|||.||||||||.|.||||:|:|:.|||||||..|.|:.|:.|||||.|||||::||.|.|||||.||||||:|||:|:|:.||.|:||||||||||||:    |:
Query: GPYGHQSGAVYVGNYKVVNRHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
	###########################################################
               

- ## sele_closest_res

In [5]:
# Load test datasets
import gemmi
from xaidar.data.molecModels import flatten_pdb
from xaidar.data.molecModels import get_pdb_stats, get_res_CoM
from xaidar.data.molecModels import  sele_pdb, sele_Lig, sele_AA, sele_closest_Chain
from xaidar.data.protocols import load_and_filter_Proteins
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")

lig_pdb = sele_pdb( ev2a_pdb, sele_Lig )
lig_com = get_res_CoM( flatten_pdb( lig_pdb, "residue") )
print( lig_com[0] )
lig_com_pos = gemmi.Position( *lig_com[0] )

ev2a_pdb  = sele_pdb( ev2a_pdb, sele_AA)
ev2a_pdb = sele_pdb( ev2a_pdb, sele_closest_Chain, lig_com_pos )
get_pdb_stats( ev2a_pdb)

[ 9.40414721 13.92196482 22.27238291]

####################
Number of models: 1
Number of chains in 1st Model: 1

Chain ID: B
	Number of Residues: 140
	Unique List of Non-A.A.: set()
	Contains A.A.


In [12]:
import numpy as np
def sele_closest_res( lst_res: list[gemmi.Residue], 
                       CoM: gemmi.Position , verbose = False, 
                       level = False) -> list[gemmi.Chain]:
    
    """
    Select the chain that contains the ligand and is closest to the CoM
    of the protein.
    Args:
    - lst_chains (list[gemmi.Chain]): List of chains in the PDB.
    - CoM (gemmi.Position): Center of Mass of the protein.
    - ligLabel (str, optional): Ligand residue name. Defaults to "LIG".
    Returns:
    - list[gemmi.Chain]: List containing the selected chain.
    """
    if level: return "residue"
    if len(lst_res) == 1:
        if verbose: print("Only one chain in the PDB, returning it")
        return lst_res
    else:
        # Select the chain closest to the CoM
        min_dist = float('inf')
        selected_res = None
        for res in lst_res:
            dist = np.linalg.norm( [get_res_CoM([res])[0], CoM] )
            if dist < min_dist:
                min_dist = dist
                selected_res = res
        return [selected_res]

In [14]:
lst_res = flatten_pdb(ev2a_pdb, "residue")
res = lst_res[0]
CoM1 = get_res_CoM([res])[0]
print( CoM1 )
CoM_ref = lig_com[0] 
print( CoM_ref )
print(np.linalg.norm((CoM1, CoM_ref)))
sele_res = sele_closest_res( lst_res, lig_com[0] )
print( sele_res[0] )

[18.78016329  5.61425588 24.70807715]
[ 9.40414721 13.92196482 22.27238291]
42.10726313296844
57(ASP)
